# 08 — Alternative Panel: Sep–Nov Pre-Period + Causal Impact

The existing notebook 06 used August as the pre-period baseline, resulting in only 6.5% coverage
(most users are not active in August). This notebook tests whether using the **anchor period
itself (Sep–Nov) as the pre-period** — measuring each user's mh_score before their first anchor
exposure — dramatically increases coverage while keeping Dec–May as the post-period.

It also implements **Causal Impact** (Bayesian structural time series) which operates on
aggregated weekly time series and does not require individual pre+post coverage.

**Pre-period definition:**
- Exposed users: all their Sep–Nov posts/comments **before** their first anchor comment
- Unexposed users: all their Sep–Nov posts/comments (no anchor comment, so full window)

**Post-period:** Dec 1 – May 31 (full decision season, same as notebook 06)

**Inputs:**
- `../r_gradadmissions_posts.jsonl` + `../r_gradadmissions_comments.jsonl` (raw)
- `../data/processed/exposure_labels.parquet`
- `../data/processed/anchor_posts.parquet`
- `../data/processed/user_community_breadth.parquet`
- `../models/clf_anxiety.joblib`, `clf_depression.joblib`, `clf_stress.joblib`

**Outputs:**
- `../data/processed/panel_scores_alt.parquet`
- Figures saved to `../figures/`

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from pathlib import Path
from datetime import datetime, timezone
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')

# ── CONFIG ──────────────────────────────────────────────────────────────────
SUBREDDIT = 'gradadmissions'   # change to 'mscs' for MSCS pipeline
# ────────────────────────────────────────────────────────────────────────────

ROOT      = Path('..').resolve()
DATA   = ROOT / 'data' / 'processed' / SUBREDDIT
MODEL_DIR = ROOT / 'models'
FIG_DIR   = ROOT / 'figures'

# Use NB01-cleaned files for all subreddits
POSTS_PATH    = DATA / 'posts_clean.jsonl'
COMMENTS_PATH = DATA / 'comments_clean.jsonl'

# Chronologically ordered admission cycles: cycle 1 = 2022, cycle 2 = 2023, cycle 3 = 2024
CYCLES = {
    1: {
        'anchor_start': datetime(2022,  9,  1, tzinfo=timezone.utc),
        'anchor_end':   datetime(2022, 11, 30, 23, 59, 59, tzinfo=timezone.utc),
        'post_start':   datetime(2022, 12,  1, tzinfo=timezone.utc),
        'post_end':     datetime(2023,  5, 31, 23, 59, 59, tzinfo=timezone.utc),
    },
    2: {
        'anchor_start': datetime(2023,  9,  1, tzinfo=timezone.utc),
        'anchor_end':   datetime(2023, 11, 30, 23, 59, 59, tzinfo=timezone.utc),
        'post_start':   datetime(2023, 12,  1, tzinfo=timezone.utc),
        'post_end':     datetime(2024,  5, 31, 23, 59, 59, tzinfo=timezone.utc),
    },
    3: {
        'anchor_start': datetime(2024,  9,  1, tzinfo=timezone.utc),
        'anchor_end':   datetime(2024, 11, 30, 23, 59, 59, tzinfo=timezone.utc),
        'post_start':   datetime(2024, 12,  1, tzinfo=timezone.utc),
        'post_end':     datetime(2025,  5, 31, 23, 59, 59, tzinfo=timezone.utc),
    },
}
CYCLE_KEYS = sorted(CYCLES)

print(f'Subreddit: r/{SUBREDDIT}')
print('Setup complete. Checking paths...')
for p in [POSTS_PATH, COMMENTS_PATH,
          DATA / 'exposure_labels.parquet',
          DATA / 'anchor_posts.parquet',
          DATA / 'user_community_breadth.parquet']:
    print(f'  {p.name}: {"OK" if p.exists() else "MISSING"}')

## 1) Load exposure labels + first anchor comment date per exposed user

In [17]:
exposure = pd.read_parquet(DATA / 'exposure_labels.parquet')
anchor_posts_df = pd.read_parquet(DATA / 'anchor_posts.parquet', columns=['id', 'cycle'])
breadth = pd.read_parquet(DATA / 'user_community_breadth.parquet',
                          columns=['author', 'community_breadth', 'community_breadth_log'])

panel_users = set(exposure['author'])
anchor_ids  = set(anchor_posts_df['id'].astype(str))

print(f'Panel users: {len(panel_users):,}')
print(f'Anchor post IDs: {len(anchor_ids):,}')
print(exposure.groupby(['cycle', 'exposed']).size())

Panel users: 20,932
Anchor post IDs: 597
cycle  exposed
1      False       9072
       True         835
2      False      10625
       True        1198
dtype: int64


In [18]:
# Find each exposed user's first anchor comment timestamp
# This defines the cutoff: pre = Sep-Nov BEFORE this date
print('Scanning comments for first anchor comment per exposed user...')

first_anchor_comment = {}  # author -> datetime of first anchor comment

with open(COMMENTS_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        r = json.loads(line)
        author  = r.get('author')
        post_id = r.get('post_id', '')
        if author not in panel_users or post_id not in anchor_ids:
            continue
        dt = datetime.fromisoformat(r['created_dt'])
        if author not in first_anchor_comment or dt < first_anchor_comment[author]:
            first_anchor_comment[author] = dt

print(f'First anchor comment found for {len(first_anchor_comment):,} exposed users')

Scanning comments for first anchor comment per exposed user...
First anchor comment found for 2,266 exposed users


## 2) Load SVM classifiers

In [19]:
clf_anx = joblib.load(MODEL_DIR / 'clf_anxiety.joblib')
clf_dep = joblib.load(MODEL_DIR / 'clf_depression.joblib')
clf_str = joblib.load(MODEL_DIR / 'clf_stress.joblib')

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def score_texts(texts):
    if not texts:
        return np.array([]), np.array([]), np.array([]), np.array([])
    anx  = sigmoid(clf_anx.decision_function(texts))
    dep  = sigmoid(clf_dep.decision_function(texts))
    str_ = sigmoid(clf_str.decision_function(texts))
    mean = np.stack([anx, dep, str_], axis=1).mean(axis=1)
    return anx, dep, str_, mean

print('Classifiers loaded.')

Classifiers loaded.


## 3) Score pre-period (Sep–Nov before first anchor) and post-period (Dec–May)

Single pass through both raw JSONL files.

In [20]:
def assign_window(author, dt, cycle):
    """
    Returns 'pre', 'post', or None.
    - Pre:  within Sep-Nov of the cycle AND before first anchor comment (if exposed)
    - Post: within Dec-May of the cycle
    """
    w = CYCLES[cycle]
    # Post window
    if w['post_start'] <= dt <= w['post_end']:
        return 'post'
    # Pre window: Sep-Nov, before first anchor comment
    if w['anchor_start'] <= dt <= w['anchor_end']:
        cutoff = first_anchor_comment.get(author)  # None if unexposed
        if cutoff is None or dt < cutoff:
            return 'pre'
    return None

# Determine which cycle(s) each user belongs to
user_cycles = exposure.groupby('author')['cycle'].apply(list).to_dict()

records = []

def process_file(path, text_field):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            author = r.get('author')
            if author not in panel_users:
                continue
            text = r.get(text_field, '') or ''
            if len(text.strip()) < 10:
                continue
            try:
                dt = datetime.fromisoformat(r['created_dt'])
            except Exception:
                continue
            for cycle in user_cycles.get(author, []):
                window = assign_window(author, dt, cycle)
                if window:
                    records.append({
                        'author': author,
                        'cycle':  cycle,
                        'window': window,
                        'dt':     dt,
                        'text':   text,
                    })

print('Processing posts...')
process_file(POSTS_PATH, 'clean_text')
n_posts = len(records)
print(f'  {n_posts:,} post records')

print('Processing comments...')
process_file(COMMENTS_PATH, 'clean_text')
print(f'  {len(records) - n_posts:,} comment records')
print(f'Total: {len(records):,}')

Processing posts...
  25,077 post records
Processing comments...
  145,343 comment records
Total: 170,420


In [21]:
corpus = pd.DataFrame(records)
print('Window distribution:')
print(corpus.groupby(['cycle', 'window']).size())

print(f'\nUnique authors in corpus: {corpus["author"].nunique():,}')
print(f'Scoring {len(corpus):,} records...')

texts = corpus['text'].tolist()
corpus['anx'], corpus['dep'], corpus['str_'], corpus['mh_score'] = score_texts(texts)
print('Scoring done.')
print(corpus['mh_score'].describe().round(4))

Window distribution:
cycle  window
1      post      57678
       pre       25253
2      post      59167
       pre       28322
dtype: int64

Unique authors in corpus: 20,043
Scoring 170,420 records...
Scoring done.
count    170420.0000
mean          0.4276
std           0.0790
min           0.0852
25%           0.3755
50%           0.4275
75%           0.4778
max           0.8302
Name: mh_score, dtype: float64


## 4) Aggregate to (author, cycle, window) → pre/post scores

In [22]:
agg = (
    corpus.groupby(['author', 'cycle', 'window'])
    .agg(mh_score=('mh_score', 'mean'), n_posts=('mh_score', 'count'))
    .reset_index()
)

pre  = agg[agg['window'] == 'pre'].drop(columns='window').rename(
    columns={'mh_score': 'pre_mh_score', 'n_posts': 'pre_n_posts'})
post = agg[agg['window'] == 'post'].drop(columns='window').rename(
    columns={'mh_score': 'post_mh_score', 'n_posts': 'post_n_posts'})

scores = pre.merge(post, on=['author', 'cycle'], how='inner')
print(f'Users with both pre AND post observations: {len(scores):,}')
print(f'  (vs {1368:,} in notebook 06 with August pre-period)')

# Merge with exposure labels and breadth
panel = exposure.merge(scores, on=['author', 'cycle'], how='inner')
panel = panel.merge(breadth, on='author', how='left')

print(f'\nFinal panel: {len(panel):,} rows, {panel["author"].nunique():,} users')
print(f'Coverage: {100 * panel["author"].nunique() / len(panel_users):.1f}% of panel users')
print('\nExposure breakdown:')
print(panel.groupby(['cycle', 'exposed']).size())

Users with both pre AND post observations: 7,868
  (vs 1,368 in notebook 06 with August pre-period)

Final panel: 7,868 rows, 7,644 users
Coverage: 36.5% of panel users

Exposure breakdown:
cycle  exposed
1      False      3403
       True        330
2      False      3707
       True        428
dtype: int64


In [23]:
# Pre-period score distribution check
print('Pre-period mh_score by exposure status:')
print(panel.groupby('exposed')['pre_mh_score'].describe().round(4))
print('\nPost-period mh_score by exposure status:')
print(panel.groupby('exposed')['post_mh_score'].describe().round(4))

panel.to_parquet(DATA / 'panel_scores_alt.parquet', index=False)
print('\nSaved panel_scores_alt.parquet')

Pre-period mh_score by exposure status:
          count    mean     std     min     25%     50%     75%     max
exposed                                                                
False    7110.0  0.4053  0.0692  0.1380  0.3641  0.4065  0.4460  0.7498
True      758.0  0.4161  0.0529  0.2087  0.3869  0.4148  0.4448  0.6034

Post-period mh_score by exposure status:
          count    mean     std     min     25%     50%     75%     max
exposed                                                                
False    7110.0  0.4284  0.0566  0.1467  0.3982  0.4309  0.4605  0.7784
True      758.0  0.4358  0.0478  0.2357  0.4109  0.4351  0.4596  0.6676

Saved panel_scores_alt.parquet


## 5) Propensity Score Matching

In [ ]:
from sklearn.neighbors import NearestNeighbors

def psm_match(df, caliper=0.05):
    """1:1 nearest-neighbor PSM with caliper on propensity score."""
    features = ['pre_mh_score', 'log1p_pre_n_posts']
    if 'community_breadth_log' in df.columns and df['community_breadth_log'].notna().mean() > 0.8:
        features.append('community_breadth_log')

    df = df.copy()
    df['log1p_pre_n_posts'] = np.log1p(df['pre_n_posts'])
    df = df.dropna(subset=features + ['exposed'])

    X = df[features].values
    y = df['exposed'].astype(int).values

    lr = LogisticRegression(max_iter=1000).fit(X, y)
    df['pscore'] = lr.predict_proba(X)[:, 1]

    treated   = df[df['exposed'] == True].copy()
    control   = df[df['exposed'] == False].copy()

    nn = NearestNeighbors(n_neighbors=1, algorithm='ball_tree')
    nn.fit(control[['pscore']].values)
    distances, indices = nn.kneighbors(treated[['pscore']].values)

    matched_treated = []
    matched_control = []
    used_control    = set()

    for i, (dist, idx) in enumerate(zip(distances.flatten(), indices.flatten())):
        ctrl_iloc = idx
        if dist <= caliper and ctrl_iloc not in used_control:
            matched_treated.append(treated.iloc[i])
            matched_control.append(control.iloc[ctrl_iloc])
            used_control.add(ctrl_iloc)

    if not matched_treated:
        print('  No matches found within caliper!')
        return pd.DataFrame()

    matched = pd.concat([
        pd.DataFrame(matched_treated),
        pd.DataFrame(matched_control)
    ], ignore_index=True)

    n_pairs = len(matched_treated)
    print(f'  {n_pairs:,} matched pairs ({len(matched):,} users kept)')

    # Balance check
    for feat in features:
        smd_before = abs(df[df['exposed']]['pre_mh_score'].mean() -
                         df[~df['exposed']]['pre_mh_score'].mean()) / df['pre_mh_score'].std()
        t_vals = matched[matched['exposed']][feat]
        c_vals = matched[~matched['exposed']][feat]
        smd_after = abs(t_vals.mean() - c_vals.mean()) / (matched[feat].std() + 1e-9)
        status = '✓' if smd_after < 0.1 else '✗'
        print(f'    {feat:<30} SMD after={smd_after:.3f} {status}')

    return matched


panel_cycles = sorted(int(c) for c in panel['cycle'].dropna().unique())
matched_by_cycle = {}
for cycle in panel_cycles:
    sub = panel[panel['cycle'] == cycle].copy()
    print(f'Cycle {cycle}: {sub["exposed"].sum():,} exposed, {(~sub["exposed"]).sum():,} unexposed')
    matched_by_cycle[cycle] = psm_match(sub)

all_matched = pd.concat(matched_by_cycle.values(), ignore_index=True)
print(f'\nTotal matched: {len(all_matched):,} rows  |  {all_matched["exposed"].sum():,} exposed')

## 6) Parallel Trends Check

In [ ]:
print('Pre-period mh_score (post-matching):')
print(all_matched.groupby(['cycle', 'exposed'])['pre_mh_score'].agg(['mean', 'std']).round(4))

n_cycles = len(panel_cycles)
fig, axes = plt.subplots(1, n_cycles, figsize=(6 * n_cycles, 4), sharey=True, squeeze=False)
axes = axes[0]
for i, cycle in enumerate(panel_cycles):
    sub = matched_by_cycle.get(cycle, pd.DataFrame())
    if sub.empty:
        continue
    for exp_val, label, color in [(True, 'Exposed', 'firebrick'), (False, 'Unexposed', 'steelblue')]:
        g = sub[sub['exposed'] == exp_val]
        axes[i].bar([0, 1], [g['pre_mh_score'].mean(), g['post_mh_score'].mean()],
                    width=0.3, label=label, alpha=0.7,
                    color=color, align='center' if exp_val else 'edge')
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(['Pre (Sep–Nov)', 'Post (Dec–May)'])
    if i == 0:
        axes[i].set_ylabel('Mean mh_score')
    axes[i].set_title(f'Cycle {cycle} — Matched Panel (r/{SUBREDDIT})')
    axes[i].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / f'fig_parallel_trends_alt_{SUBREDDIT}.png', dpi=150, bbox_inches='tight')
plt.show()

## 7) DiD Regression — RQ1 + RQ2

In [ ]:
def run_did(matched, cycle_label='Pooled'):
    if matched.empty or matched['exposed'].sum() < 10:
        print(f'{cycle_label}: insufficient data')
        return None, None

    # Reshape to long format: one row per (user, period)
    pre_rows  = matched[['author', 'cycle', 'exposed', 'pre_mh_score',  'pre_n_posts',  'community_breadth_log']].copy()
    post_rows = matched[['author', 'cycle', 'exposed', 'post_mh_score', 'post_n_posts', 'community_breadth_log']].copy()

    pre_rows  = pre_rows.rename(columns={'pre_mh_score': 'mh_score', 'pre_n_posts': 'n_posts'})
    post_rows = post_rows.rename(columns={'post_mh_score': 'mh_score', 'post_n_posts': 'n_posts'})

    pre_rows['period']  = 0
    post_rows['period'] = 1

    long = pd.concat([pre_rows, post_rows], ignore_index=True)
    long['exposed']     = long['exposed'].astype(int)
    long['log1p_posts'] = np.log1p(long['n_posts'])
    long['period_x_exposed'] = long['period'] * long['exposed']

    # RQ1
    m1 = smf.ols('mh_score ~ period + exposed + period_x_exposed + log1p_posts',
                 data=long).fit(cov_type='HC3')
    coef = m1.params['period_x_exposed']
    ci   = m1.conf_int().loc['period_x_exposed']
    p    = m1.pvalues['period_x_exposed']
    sig  = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
    print(f'{cycle_label:<20} DiD={coef:+.4f} [{ci[0]:+.4f}, {ci[1]:+.4f}]  p={p:.4f} {sig}  n={len(long):,}')

    # RQ2 — breadth moderation
    rq2 = None
    if long['community_breadth_log'].notna().mean() > 0.5:
        long2 = long.dropna(subset=['community_breadth_log']).copy()
        long2['period_x_exposed_x_breadth'] = long2['period_x_exposed'] * long2['community_breadth_log']
        m2 = smf.ols(
            'mh_score ~ period + exposed + period_x_exposed + log1p_posts + '
            'community_breadth_log + period_x_exposed_x_breadth',
            data=long2
        ).fit(cov_type='HC3')
        b3   = m2.params['period_x_exposed_x_breadth']
        ci3  = m2.conf_int().loc['period_x_exposed_x_breadth']
        p3   = m2.pvalues['period_x_exposed_x_breadth']
        sig3 = '***' if p3 < 0.001 else '**' if p3 < 0.01 else '*' if p3 < 0.05 else 'n.s.'
        direction = 'buffers' if b3 < 0 else 'amplifies'
        print(f'  RQ2 breadth moderation: {b3:+.4f} [{ci3[0]:+.4f}, {ci3[1]:+.4f}]  '
              f'p={p3:.4f} {sig3}  ({direction})')
        rq2 = m2

    return m1, rq2


print('=== RQ1 + RQ2: DiD on mh_score (HC3 SEs) ===')
results = {}
for cycle in panel_cycles:
    results[cycle] = run_did(matched_by_cycle[cycle], f'Cycle {cycle}')

# Add cycle FE for pooled
print()
pooled = all_matched.copy()
pre_p  = pooled[['author','cycle','exposed','pre_mh_score','pre_n_posts','community_breadth_log']].rename(
    columns={'pre_mh_score':'mh_score','pre_n_posts':'n_posts'})
post_p = pooled[['author','cycle','exposed','post_mh_score','post_n_posts','community_breadth_log']].rename(
    columns={'post_mh_score':'mh_score','post_n_posts':'n_posts'})
pre_p['period'] = 0; post_p['period'] = 1
long_p = pd.concat([pre_p, post_p], ignore_index=True)
long_p['exposed'] = long_p['exposed'].astype(int)
long_p['log1p_posts'] = np.log1p(long_p['n_posts'])
long_p['period_x_exposed'] = long_p['period'] * long_p['exposed']
m_pool = smf.ols('mh_score ~ period + exposed + period_x_exposed + log1p_posts + C(cycle)',
                 data=long_p).fit(cov_type='HC3')
coef = m_pool.params['period_x_exposed']
ci   = m_pool.conf_int().loc['period_x_exposed']
p    = m_pool.pvalues['period_x_exposed']
sig  = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
print(f'Pooled (+cycle FE)    DiD={coef:+.4f} [{ci[0]:+.4f}, {ci[1]:+.4f}]  p={p:.4f} {sig}  n={len(long_p):,}')

## 8) Causal Impact — Bayesian structural time series

Uses aggregated **weekly** mean mh_score for exposed vs unexposed groups across the full Aug–May window.
Does not require individual pre+post coverage — all panel users contribute.

- Pre-intervention: Sep 1 – Nov 30 (anchor period)
- Intervention point: Dec 1 (start of decision season)
- Post-intervention: Dec 1 – May 31

In [27]:
from causalimpact import CausalImpact

# causalimpact 0.2.6 uses a pandas function removed in pandas 2.x — patch it back in
import pandas.core.dtypes.common as _pdc
if not hasattr(_pdc, 'is_datetime_or_timedelta_dtype'):
    _pdc.is_datetime_or_timedelta_dtype = (
        lambda x: pd.api.types.is_datetime64_any_dtype(x) or
                  pd.api.types.is_timedelta64_dtype(x)
    )

# Build weekly time series from corpus
corpus['week'] = corpus['dt'].dt.to_period('W').dt.start_time

# Join exposure info on (author, cycle) — users can appear in multiple cycles
corpus_exp = corpus.merge(
    exposure[['author', 'cycle', 'exposed']],
    on=['author', 'cycle'],
    how='left'
)
corpus_exp = corpus_exp.dropna(subset=['exposed'])

weekly = (
    corpus_exp.groupby(['week', 'cycle', 'exposed'])
    ['mh_score'].mean()
    .reset_index()
    .rename(columns={'mh_score': 'mean_mh'})
)

print(f'Weekly records: {len(weekly):,}')
print(weekly.groupby('cycle')['week'].agg(['min', 'max']))

Weekly records: 160
             min        max
cycle                      
1     2023-08-28 2024-05-27
2     2024-08-26 2025-05-26


In [ ]:
# Run Causal Impact for each cycle present in the panel
for cycle in panel_cycles:
    w = CYCLES[cycle]
    post_start = pd.Timestamp(w['post_start']).tz_localize(None).normalize()
    post_end   = pd.Timestamp(w['post_end']).tz_localize(None).normalize()
    pre_start  = pd.Timestamp(w['anchor_start']).tz_localize(None).normalize()
    pre_end    = (post_start - pd.Timedelta(days=1)).normalize()

    sub = weekly[weekly['cycle'] == cycle]
    wide = sub.pivot(index='week', columns='exposed', values='mean_mh')
    wide.columns = ['unexposed', 'exposed']
    wide = wide.sort_index().dropna()
    wide.index = pd.to_datetime(wide.index)
    wide = wide[(wide.index >= pre_start) & (wide.index <= post_end)].astype(float)

    print(f'\n=== Cycle {cycle} Causal Impact ===')
    print(f'  Pre:  {pre_start.date()} → {pre_end.date()}')
    print(f'  Post: {post_start.date()} → {post_end.date()}')
    print(f'  Weeks: {len(wide)} total')

    if len(wide) < 5:
        print('  Insufficient weeks, skipping.')
        continue

    # Use integer positions — avoids causalimpact 0.2.6 / pandas 2.x datetime compat bug
    n_pre = len(wide[wide.index < post_start])
    ci_df = wide.reset_index(drop=True)
    int_pre  = [0, n_pre - 1]
    int_post = [n_pre, len(ci_df) - 1]

    try:
        ci = CausalImpact(ci_df, int_pre, int_post)
        ci.run()
        print(ci.summary())
        out_path = FIG_DIR / f'fig_causal_impact_cycle{cycle}_{SUBREDDIT}.png'
        ci.plot(figsize=(15, 12), fname=out_path)
        print(f'  Saved {out_path.name}')
    except Exception as e:
        print(f'  Causal Impact failed: {e}')

## 9) Summary comparison vs notebook 06

In [29]:
print('=== Coverage comparison ===')
print(f'Notebook 06 (Aug pre-period):     1,368 users, 322 exposed, ~155 matched pairs/cycle')
print(f'Notebook 08 (Sep-Nov pre-period):  {panel["author"].nunique():,} users, '
      f'{panel["exposed"].sum():,} exposed, see matched pairs above')
print()
print('Key methodological difference:')
print('  NB06: pre = August (low activity month → 6.5% coverage)')
print('  NB08: pre = Sep-Nov before first anchor comment (active period → higher coverage)')
print('  NB08 Causal Impact: no individual coverage required — uses aggregate weekly series')

=== Coverage comparison ===
Notebook 06 (Aug pre-period):     1,368 users, 322 exposed, ~155 matched pairs/cycle
Notebook 08 (Sep-Nov pre-period):  7,644 users, 758 exposed, see matched pairs above

Key methodological difference:
  NB06: pre = August (low activity month → 6.5% coverage)
  NB08: pre = Sep-Nov before first anchor comment (active period → higher coverage)
  NB08 Causal Impact: no individual coverage required — uses aggregate weekly series
